## 1. Environment Setup

In [ ]:
# Install / upgrade dependencies
!pip install -q gensim spacy
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import glob
import pandas as pd
from time import time
from collections import defaultdict
import multiprocessing

import spacy
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser

import logging
logging.basicConfig(
    format="%(levelname)s - %(asctime)s: %(message)s",
    datefmt='%H:%M:%S',
    level=logging.INFO
)

print("All imports OK")

All imports OK


## 2. Loading COHA Files

COHA `.txt` files have a specific markup format. We load every `.txt` file in the target folder and split on document boundaries.

In [ ]:
COHA_GLOB = "/content/*.txt"
# If running locally with the uploaded files:
# COHA_GLOB = "/mnt/user-data/uploads/*.txt"
# ──────────────────────────────────────────────────────────────────────────────

raw_docs = []
file_paths = sorted(glob.glob(COHA_GLOB))


for fp in file_paths:
    with open(fp, encoding='utf-8', errors='replace') as f:
        content = f.read()
    # Split on document-boundary markers (@@XXXX)
    parts = re.split(r'@@\d+', content)
    for part in parts:
        stripped = part.strip()
        if stripped:
            raw_docs.append(stripped)

print(f"Loaded {len(file_paths)} file(s) → {len(raw_docs)} document segment(s)")
print("\nFirst 300 chars of first segment:")
print(raw_docs[0][:300])

Loaded 4 file(s) → 4 document segment(s)

First 300 chars of first segment:
The author is indebted to one of the novels of Le Brun , for the ground-work of this little comedy . DRAMATIS PERSON . Philadelphia Count Almeyda Mr. Robertson Count Arandez Warren Carlos Wood Pacomo Jefferson Gusman Abercrombie Pedrillo Durang Herald Jackson Eugenia Mrs. Entwistle Beatrice Francis 


## 3. Cleaning & Preprocessing

We apply a multi-step cleaning pipeline tailored to COHA dramatic fiction:

1. Strip COHA redaction markers (`@ @ @ @ @ @ @ @ @ @`)
2. Remove stage directions in parentheses — `( exits hastily )`
3. Remove character name headers — COHA uses `Name . Name speech…` patterns
4. Remove section headers — `Main text`, `ACT`, `SCENE`, `DRAMATIS PERSON`
5. Keep only alphabetic tokens and lowercase
6. Lemmatize & remove stopwords via spaCy

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def coha_preprocess(text: str) -> str:
    """Strip COHA markup and dramatic-text noise from a raw document string."""
    # 1. Remove redaction markers (space-separated @ symbols)
    text = re.sub(r'(?:@\s*)+', ' ', text)

    # 2. Remove stage directions in parentheses (including nested)
    text = re.sub(r'\([^)]*\)', ' ', text)

    # 3. Remove structural headers typical of COHA dramatic texts
    headers = [
        r'Main text', r'DRAMATIS PERSON\.?', r'ACT\s+[IVXLC]+\.?',
        r'SCENE\s+[IVXLC]+', r'SCENE\s+\w+', r'CHORUS',
        r'END\s+OF\s+ACT', r'STAGE DIRECTIONS', r'EXITS? AND ENTRANCES?',
        r'RELATIVE POSITIONS?',
    ]
    for h in headers:
        text = re.sub(h, ' ', text, flags=re.IGNORECASE)

    # 4. Remove character-name prefixes: "Carlos . Carlos" or "Eug . Eugenia"
    #    Pattern: short abbreviation, dot, full name (all followed by the speech)
    text = re.sub(r'\b[A-Z][a-z]{0,6}\s*\.\s*[A-Z][a-zA-Z]+\s+', ' ', text)

    # 5. Remove ALL-CAPS single words (stage direction labels like "Aside", "R.", "L.")
    text = re.sub(r'\b[A-Z]{2,}\b\.?', ' ', text)

    # 6. Collapse whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


#spaCy cleaning (lemmatize, remove stopwords, keep only alpha) ──────────────

def spacy_clean(doc) -> str | None:
    """Lemmatize and remove stopwords/punctuation. Returns None for very short docs."""
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha and len(token.text) > 1
    ]
    if len(tokens) > 2:
        return ' '.join(tokens)
    return None


print("Preprocessing pipeline defined.")

Preprocessing pipeline defined.


In [ ]:
#COHA-specific cleaning
prepped_texts = [coha_preprocess(doc) for doc in raw_docs]

# Preview one result
print("After COHA pre-cleaning (first 300 chars):")
print(prepped_texts[0][:30])

After COHA pre-cleaning (first 300 chars):
The author is indebted to one 


In [ ]:
# ── Step 2: spaCy lemmatization ─────────────────────────────────────────────────
# We also apply a brief regex pass to remove remaining non-alpha chars
brief_clean = (
    re.sub(r"[^A-Za-z']+", ' ', text).lower()
    for text in prepped_texts
)

t = time()
cleaned = [
    spacy_clean(doc)
    for doc in nlp.pipe(brief_clean, batch_size=200)
]
print(f"spaCy cleaning took {round((time() - t) / 60, 2)} min")

# Collect into a DataFrame and drop nulls / duplicates
df_clean = pd.DataFrame({'clean': cleaned}).dropna().drop_duplicates().reset_index(drop=True)
print(f"Clean segments: {len(df_clean)}")
df_clean.head()

spaCy cleaning took 0.05 min
Clean segments: 4


,clean
0,author indebted novel le brun ground work litt...
1,alexius emperour bouillon duke lorraine bohemo...
2,oh love world make fatal love ah cypress branc...
3,feeble sincere testimonial respect entertain o...


In [ ]:
# Quick sanity check — most common tokens
from collections import Counter
all_words = ' '.join(df_clean['clean']).split()
print("Total tokens aftezr cleaning:", len(all_words))
print("\nTop 20 most common tokens:")
Counter(all_words).most_common(20)

Total tokens aftezr cleaning: 12022

Top 20 most common tokens:


[('almeyda', 152),
 ('shall', 87),
 ('sir', 85),
 ('come', 81),
 ('ziani', 73),
 ('count', 69),
 ('know', 68),
 ('thy', 64),
 ('arandez', 61),
 ('like', 60),
 ('enter', 60),
 ('love', 59),
 ('man', 58),
 ('master', 58),
 ('let', 55),
 ('father', 54),
 ('good', 53),
 ('demetri', 53),
 ('lady', 52),
 ('yes', 51)]

## 4. Bigrams

We use Gensim's `Phrases` to detect commonly co-occurring word pairs.
In a historical drama corpus this captures expressions like `count_almeyda`, `evil_eye`, `holy_land`, etc.

In [ ]:
# Convert each cleaned segment to a list of words
sent = [row.split() for row in df_clean['clean']]
print(f"{len(sent)} sentence lists")

# ── Bigram detection ───────────────────────────────────────────────────────────
# min_count is lower than the Simpsons tutorial because COHA sample is smaller.
# Increase to 10–30 when using the full COHA corpus.
phrases = Phrases(sent, min_count=3, threshold=5, progress_per=500)
bigram  = Phraser(phrases)
sentences = list(bigram[sent])

# Show detected bigrams
bigrams_found = [
    token for seg in sentences for token in seg if '_' in token
]
print(f"\nExample bigrams detected ({len(set(bigrams_found))} unique):")
print(sorted(set(bigrams_found))[:30])

4 sentence lists

Example bigrams detected (36 unique):
['ah_sir', 'arandez_act', 'art_thou', 'come_hither', 'count_almeyda', 'count_arandez', 'count_thoulouse', 'crow_bar', 'cyril_ziani', 'enter_castle', 'evil_eye', 'fair_sultana', 'flora_flora', 'good_night', 'katusthius_ziani', 'kilidge_arslan', 'mountain_home', 'old_gentleman', 'old_man', 'old_woman', 'poor_pacomo', 'private_stair', 'raymond_count', 'run_away', 'sultan_kilidge', 'sultan_solyman', 'thou_art', 'wall_antioch', 'year_ago', 'yes_signor']


In [ ]:
# Word frequency distribution
word_freq = defaultdict(int)
for seg in sentences:
    for w in seg:
        word_freq[w] += 1

print(f"Vocabulary size (before min_count filter): {len(word_freq)}")
print("\nTop 15 tokens:")
sorted(word_freq, key=word_freq.get, reverse=True)[:15]

Vocabulary size (before min_count filter): 3386

Top 15 tokens:


['almeyda',
 'shall',
 'come',
 'know',
 'sir',
 'thy',
 'like',
 'love',
 'master',
 'enter',
 'let',
 'father',
 'thee',
 'tancre',
 'boy']

## 5. Training the Model

### Parameter notes for COHA

| Parameter | Simpsons value | COHA (sample) value | Rationale |
|---|---|---|---|
| `min_count` | 20 | **5** | Smaller corpus → keep rarer words |
| `window` | 2 | **5** | Longer sentences in formal prose |
| `vector_size` | 300 | **100** | Proportional to corpus size |
| `sample` | 6e-5 | **1e-4** | Less aggressive downsampling |
| `epochs` | 30 | **30** | Same |

> When you scale to the full COHA (~400 million words) you can safely raise `min_count` to 20, `vector_size` to 300, and lower `sample` to 6e-5 as in the original tutorial.

In [ ]:
cores = multiprocessing.cpu_count()
print(f"CPU cores available: {cores}")

w2v_model = Word2Vec(
    min_count   = 5,
    window      = 5,
    vector_size = 100,
    sample      = 1e-4,
    alpha       = 0.03,
    min_alpha   = 0.0007,
    negative    = 20,
    workers     = max(1, cores - 1)
)

CPU cores available: 2


In [ ]:
#build the vocabulary
t = time()
w2v_model.build_vocab(sentences, progress_per=1000)
print(f"Vocabulary built in {round((time() - t) / 60, 2)} min")
print(f"Vocabulary size: {len(w2v_model.wv)}")

Vocabulary built in 0.0 min
Vocabulary size: 617


In [ ]:
#train the model
t = time()
w2v_model.train(
    sentences,
    total_examples = w2v_model.corpus_count,
    epochs         = 30,
    report_delay   = 1
)
print(f"Model trained in {round((time() - t) / 60, 2)} min")

Model trained in 0.01 min


In [ ]:
#lock the vectors (memory-efficient after training)
w2v_model.wv.vectors_lockf = 0.0   # Gensim 4.x replacement for init_sims(replace=True)

#save for later use
w2v_model.save("coha_word2vec.model")
print("Model saved as coha_word2vec.model")

Model saved as coha_word2vec.model


## 6. Exploring the Model

substitute Simpsons character queries with vocabulary appropriate to early 19th-century dramatic prose.

### Most-similar words

In [ ]:
#helper to print results cleanly
def show_similar(word, topn=10):
    try:
        results = w2v_model.wv.most_similar(positive=[word], topn=topn)
        print(f"\nMost similar to '{word}':")
        for w, score in results:
            print(f"  {w:<20} {score:.4f}")
    except KeyError:
        print(f"'{word}' not in vocabulary — try another word.")

In [ ]:
show_similar("love")


Most similar to 'love':
  betray               0.9997
  think                0.9997
  need                 0.9996
  ear                  0.9996
  league               0.9996
  son                  0.9996
  hear                 0.9996
  character            0.9996
  discover             0.9996
  bold                 0.9996


In [ ]:
show_similar("fate")


Most similar to 'fate':
  wait                 0.9996
  lie                  0.9996
  lodge                0.9995
  surely               0.9995
  gallant              0.9995
  menkatiz             0.9995
  friendship           0.9995
  short                0.9995
  subject              0.9995
  need                 0.9995


In [ ]:
show_similar("honour")


Most similar to 'honour':
  pardon               0.9996
  near                 0.9996
  perceive             0.9996
  leave                0.9996
  keeper               0.9996
  happiness            0.9996
  learn                0.9996
  tell                 0.9996
  death                0.9996
  league               0.9996


In [ ]:
# Try any word from the vocabulary
show_similar("heart")
show_similar("villain")


Most similar to 'heart':
  freely               0.9997
  like                 0.9997
  join                 0.9996
  fast                 0.9996
  fall                 0.9996
  present              0.9996
  think                0.9996
  force                0.9996
  friendship           0.9996
  suppose              0.9996

Most similar to 'villain':
  compose              0.9996
  seek                 0.9996
  return               0.9996
  son                  0.9996
  lover                0.9996
  steal                0.9996
  courage              0.9996
  place                0.9996
  character            0.9996
  peace                0.9996


### Similarity scores between pairs

In [ ]:
#liste de paires de mots pour lesquelles on veut mesurer la similarité
#chaque tuple contient (mot1, mot2)
pairs = [
    ("love",    "passion"),
    ("love",    "hatred"),
    ("father",  "daughter"),
    ("sword",   "honour"),
    ("villain", "hero"),
]

#affichage de l’en-tête du tableau
# {:<30} = aligné à gauche sur 30 caractères
# {:>10} = aligné à droite sur 10 caractères
print(f"{'Pair':<30} {'Similarity':>10}")
print("-" * 42)

#boucle sur chaque paire de mots
for w1, w2 in pairs:
    try:
        #calcul de la similarité cosinus entre les deux mots
        #w2v_model.wv.similarity() renvoie une valeur entre -1 et 1
        sim = w2v_model.wv.similarity(w1, w2)

        #affichage formaté : mot1 / mot2 + score arrondi à 4 décimales
        print(f"{w1} / {w2:<25} {sim:>10.4f}")

    except KeyError as e:
        # Si un mot n’est pas dans le vocabulaire du modèle Word2Vec
        print(f"{w1} / {w2:<25} {str(e)} not in vocab")


### Odd-one-out

In [ ]:
# Define a function to find the "odd one out"
def odd_one_out(words):
    #keep only words that exist in the model vocabulary
    in_vocab = [w for w in words if w in w2v_model.wv]

    #ensure there are enough valid words for comparison
    if len(in_vocab) < 3:
        print(f"Need at least 3 in-vocab words; found: {in_vocab}")
        return

    #find the word least similar to the others
    result = w2v_model.wv.doesnt_match(in_vocab)
    print(f"Odd one out from {in_vocab}: '{result}'")

# Example 1:
# "sword" is expected to differ from emotion-related words
odd_one_out([
    "love",
    "passion",
    "heart",
    "sword"
])

# Example 2:
# "enemy" is expected to differ from family-related words
odd_one_out([
    "father",
    "mother",
    "daughter",
    "enemy"
])

### Analogy — vector arithmetic

Classic Word2Vec analogy: *king − man + woman ≈ queen*

In the COHA drama context we can try:
- *daughter − father + mother ≈ ?*
- *honour − man + woman ≈ ?*

In [ ]:
# Define a helper function for word analogies
def analogy(positive, negative, topn=3):
    """
    positive : list
        Words that contribute positively to the analogy.

    negative : list
        Words that are subtracted from the analogy.

    topn : int
        Number of most similar results to return.
    """

    try:
        #compute the most similar words using vector arithmetic
        results = w2v_model.wv.most_similar(
            positive=positive,
            negative=negative,
            topn=topn
        )

        #create a readable equation string
        eq = " + ".join(positive) + " - " + " - ".join(negative)

        #display the analogy expression
        print(f"\n{eq}  →")

        #print each result with its similarity score
        for w, score in results:
            print(f"  {w:<20} {score:.4f}")

    #handle missing words in the vocabulary
    except KeyError as e:
        print(f"KeyError: {e}")


# Example 1:
# daughter + mother - father
# Tests family relationship embeddings
analogy(
    positive=["daughter", "mother"],
    negative=["father"]
)

# Example 2:
# honour + woman - man
# Explores gender associations in embeddings
analogy(
    positive=["honour", "woman"],
    negative=["man"]
)

## 7. t-SNE Visualizations

We reduce the `vector_size`-dimensional embeddings to 2D using PCA + t-SNE and plot:
- The **query word** in red
- Its **10 most similar** words in blue
- A **comparison list** in green

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#tools for dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

#display plots directly inside the notebook
%matplotlib inline

#set Seaborn plot style
sns.set_style("darkgrid")


def tsnescatterplot(model, word, list_names, title=None):
    """
    Create a t-SNE visualization for:
    - the target word (shown in red)
    - its top 10 most similar words (shown in blue)
    - additional comparison words from `list_names` (shown in green)
    """

    #get the embedding vector size from the Word2Vec model
    dim = model.wv.vector_size

    #initialize arrays and metadata containers
    arrays = np.empty((0, dim), dtype='f')
    word_labels = [word]
    color_list = ['red']

    #add the vector for the target/query word
    arrays = np.append(arrays, model.wv[word].reshape(1, -1), axis=0)

    #retrieve and add the top 10 most similar words
    for wrd, _ in model.wv.most_similar([word], topn=10):

        #append the word vector
        arrays = np.append(arrays, model.wv[wrd].reshape(1, -1), axis=0)

        #store label and display color
        word_labels.append(wrd)
        color_list.append('blue')

    #add comparison words from the provided list
    #ignore words that are not in the vocabulary (OOV = Out Of Vocabulary)
    for wrd in list_names:
        if wrd in model.wv:

            #append the word vector
            arrays = np.append(arrays, model.wv[wrd].reshape(1, -1), axis=0)

            #store label and display color
            word_labels.append(wrd)
            color_list.append('green')

    # Dimensionality reduction

    # First reduce dimensions using PCA
    # Limit PCA dimensions to:
    # 50 maximum
    # number of samples - 1
    # original embedding dimension
    n_pca = min(50, arrays.shape[0] - 1, dim)

    reduc = PCA(n_components=n_pca).fit_transform(arrays)

    #set perplexity for t-SNE
    #must be smaller than the number of samples
    perp = min(15, reduc.shape[0] - 1)

    #reduce to 2D using t-SNE
    Y = TSNE(
        n_components=2,
        random_state=42,
        perplexity=perp
    ).fit_transform(reduc)

    #create a DataFrame for plotting
    df_plot = pd.DataFrame({
        'x': Y[:, 0],
        'y': Y[:, 1],
        'words': word_labels,
        'color': color_list
    })

    # Plotting

    #create figure and axes
    fig, ax = plt.subplots(figsize=(10, 10))

    #draw scatter plot
    sns.scatterplot(
        data=df_plot,
        x='x',
        y='y',
        hue='color',
        palette={
            'red': 'red',
            'blue': 'royalblue',
            'green': 'seagreen'
        },
        s=60,
        legend=False,
        ax=ax
    )

    #add text labels next to each point
    for _, row in df_plot.iterrows():
        ax.text(
            row['x'] + 0.5,
            row['y'],
            row['words'].title(),
            fontsize=11,
            color=row['color'],
            weight='normal'
        )

    #set plot title
    ax.set_title(title or f't-SNE — "{word}"', fontsize=14)

    #improve spacing
    plt.tight_layout()

    #display the plot
    plt.show()


#confirmation message
print("tsnescatterplot() defined.")

### Plot 1 — `love` vs. random semantic neighbours

In [ ]:
tsnescatterplot(
    w2v_model,
    word       = 'love',
    list_names = ['sword', 'death', 'villain', 'blood', 'castle', 'father', 'honour'],
    title      = 't-SNE — "love" (blue: top-10 similar | green: semantic contrasts)'
)

### Plot 2 — `fate` vs. its 10 most dissimilar words

In [ ]:
tsnescatterplot(
    w2v_model,
    word       = 'fate',
    list_names = [w for w, _ in w2v_model.wv.most_similar(negative=['fate'], topn=8)],
    title      = 't-SNE — "fate" (blue: most similar | green: most dissimilar)'
)

### Plot 3 — `honour`: top 1–10 vs. top 11–20 similar words

In [ ]:
#try using the British spelling "honour"
try:
    # Get the 20 most similar words to "honour"
    top20 = [w for w, _ in w2v_model.wv.most_similar(positive=['honour'], topn=20)]

    # Visualize the word and its similar words with t-SNE
    tsnescatterplot(
        w2v_model,
        word='honour',
        list_names=top20[10:],  # Display words ranked 11–20
        title='t-SNE — "honour" (blue: top 1-10 | green: top 11-20)'
    )

# If "honour" is not found in the vocabulary,
# fall back to the American spelling "honor"
except KeyError:

    # Get the 20 most similar words to "honor"
    top20 = [w for w, _ in w2v_model.wv.most_similar(positive=['honor'], topn=20)]

    # Visualize the results for "honor"
    tsnescatterplot(
        w2v_model,
        'honor',
        top20[10:]
    )